# Chapter 18: with, match, and else Blocks
*From: Fluent Python by Luciano Ramalho*

---

This chapter covers three control flow features that are underused in Python: the **with** statement and context managers, advanced **match/case** patterns, and the **else** clause on `for`, `while`, and `try` statements.

## TL;DR

- The **with** statement invokes `__enter__` at the start and `__exit__` at the end, guaranteeing teardown even on exceptions.
- The **context manager** object is distinct from the **as** value (`__enter__`'s return).
- **`@contextlib.contextmanager`** turns a generator with a single `yield` into a context manager -- wrap the `yield` in `try/finally` for safe cleanup.
- **`contextlib`** provides `closing`, `suppress`, `nullcontext`, `ExitStack`, and more.
- **match/case** handles class patterns, sequence patterns with guards, OR-patterns (`a | b`), and nested destructuring.
- **else on for/while**: runs if the loop completes without `break`.
- **else on try**: runs only if no exception was raised -- separates guarded code from follow-up.

---
## 1. Context Managers and the with Statement

The `with` statement simplifies `try/finally` by guaranteeing that cleanup code runs, even if exceptions occur.

The context manager protocol requires two methods:
- `__enter__`: set up the resource, return the "as" value
- `__exit__(exc_type, exc_value, traceback)`: tear down, optionally suppress exceptions

**Key distinction:** the context manager object (after `with`) is NOT the same as the `as` value (`__enter__`'s return).

In [ ]:
# File objects are the most common context managers
import tempfile, os

# Create a temp file for demonstration
tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False)
tmp_name = tmp.name
tmp.close()

# Write using with -- file is guaranteed to close
with open(tmp_name, "w") as fp:
    fp.write("Hello from with block!")
    print(f"Inside with: fp.closed = {fp.closed}")

print(f"After with:  fp.closed = {fp.closed}")
print(f"fp variable still accessible: {fp.name}")

# Read back
with open(tmp_name) as fp:
    print(f"Contents: {fp.read()}")

os.unlink(tmp_name)

In [ ]:
# Custom context manager class
import sys

class LookingGlass:
    """Context manager that reverses stdout output."""
    def __enter__(self):
        self.original_write = sys.stdout.write
        sys.stdout.write = self.reverse_write
        return "JABBERWOCKY"  # this is the "as" value

    def reverse_write(self, text):
        self.original_write(text[::-1])

    def __exit__(self, exc_type, exc_value, traceback):
        sys.stdout.write = self.original_write
        if exc_type is ZeroDivisionError:
            print("Please DO NOT divide by zero!")
            return True  # suppress the exception

# The "as" value is what __enter__ returns
with LookingGlass() as what:
    print("Alice, Kitty and Snowdrop")
    print(what)

# Back to normal
print(f"what = {what!r}")

In [ ]:
# __exit__ can suppress exceptions by returning True
with LookingGlass():
    print("This is reversed")
    # 1 / 0  # Would be caught and suppressed!

---
## 2. The contextlib Utilities

The `contextlib` module provides several helpers:

| Utility | Purpose |
|---|---|
| `closing(obj)` | Wraps objects with `.close()` as context managers |
| `suppress(*exc)` | Temporarily ignore specified exceptions |
| `nullcontext(value)` | No-op context manager (for conditional use) |
| `ExitStack` | Manage a dynamic number of context managers |
| `redirect_stdout` | Redirect stdout to a file-like object |
| `@contextmanager` | Build a CM from a generator function |

In [ ]:
from contextlib import suppress, redirect_stdout
import io

# suppress: ignore specific exceptions
with suppress(FileNotFoundError):
    os.remove("/tmp/nonexistent_file_xyz.txt")
print("No crash even though file does not exist")

# redirect_stdout: capture output
buffer = io.StringIO()
with redirect_stdout(buffer):
    print("This goes to the buffer")
    print("Not to the console")

print(f"Captured: {buffer.getvalue()!r}")

---
## 3. Using @contextmanager

The `@contextlib.contextmanager` decorator turns a generator with a **single yield** into a context manager:
- Code **before** `yield` runs on `__enter__`
- The **yielded value** becomes the `as` target
- Code **after** `yield` runs on `__exit__`

**Essential:** wrap the `yield` in `try/finally` for safe cleanup.

In [ ]:
import contextlib
import sys

@contextlib.contextmanager
def looking_glass():
    original_write = sys.stdout.write
    def reverse_write(text):
        original_write(text[::-1])
    sys.stdout.write = reverse_write
    msg = ""
    try:
        yield "JABBERWOCKY"  # pauses here during the with block
    except ZeroDivisionError:
        msg = "Please DO NOT divide by zero!"
    finally:
        sys.stdout.write = original_write  # always restore!
        if msg:
            print(msg)

with looking_glass() as what:
    print("Alice, Kitty and Snowdrop")
    print(what)

print(f"Back to normal. what = {what!r}")

In [ ]:
# Practical example: temporary directory change
import os

@contextlib.contextmanager
def working_directory(path):
    """Temporarily change to a different directory."""
    old_cwd = os.getcwd()
    try:
        os.chdir(path)
        yield path
    finally:
        os.chdir(old_cwd)

original = os.getcwd()
with working_directory("/tmp") as p:
    print(f"Inside: cwd = {os.getcwd()}")
print(f"After:  cwd = {os.getcwd()}")
assert os.getcwd() == original

---
## 4. Pattern Matching Beyond Sequences

`match/case` supports class patterns, OR-patterns, guards, and nested destructuring. It is especially powerful for building language interpreters and DSLs.

In [ ]:
# match/case with different pattern types
def evaluate(expression):
    """Simple expression evaluator using match/case."""
    match expression:
        case ("add", x, y):
            return x + y
        case ("mul", x, y):
            return x * y
        case ("neg", x):
            return -x
        case int(x) | float(x):  # OR pattern with class patterns
            return x
        case _:
            raise ValueError(f"Unknown expression: {expression}")

print(evaluate(("add", 3, 4)))       # 7
print(evaluate(("mul", 5, 6)))       # 30
print(evaluate(("neg", 42)))         # -42
print(evaluate(99))                  # 99
print(evaluate(3.14))                # 3.14

In [ ]:
# Nested match/case with guards
def describe_sequence(seq):
    match seq:
        case []:
            return "empty"
        case [x]:
            return f"single element: {x}"
        case [x, y] if x == y:
            return f"pair of identical: {x}"
        case [x, y]:
            return f"pair: {x} and {y}"
        case [x, *rest] if len(rest) > 5:
            return f"long sequence starting with {x}"
        case [x, *rest]:
            return f"sequence: {x} followed by {len(rest)} more"

print(describe_sequence([]))
print(describe_sequence([42]))
print(describe_sequence([7, 7]))
print(describe_sequence([7, 8]))
print(describe_sequence([1, 2, 3, 4, 5, 6, 7]))
print(describe_sequence([1, 2, 3]))

---
## 5. OR-patterns in match/case

Subpatterns separated by `|` form an **OR-pattern** that succeeds if any alternative matches. All alternatives must bind the **same set of variables**.

In [ ]:
# OR patterns
def classify_char(char):
    match char:
        case "a" | "e" | "i" | "o" | "u":
            return "lowercase vowel"
        case "A" | "E" | "I" | "O" | "U":
            return "uppercase vowel"
        case str(c) if c.isalpha():
            return "consonant"
        case str(c) if c.isdigit():
            return "digit"
        case _:
            return "other"

for ch in "Hello World! 42":
    print(f"{ch!r:5} -> {classify_char(ch)}")

---
## 6. else Blocks Beyond if

Python allows `else` on `for`, `while`, and `try`:

| Statement | else runs when... |
|---|---|
| `for...else` | Loop completed **without** `break` |
| `while...else` | Condition became falsy (no `break`) |
| `try...else` | No exception was raised |

Think of `else` as "then" or "no break" for loops, and "no error" for try.

In [ ]:
# for...else: runs only if NO break occurred
def find_item(haystack, needle):
    for item in haystack:
        if item == needle:
            print(f"Found {needle}!")
            break
    else:
        # This runs only if the loop did NOT break
        print(f"{needle} not found in {haystack}")

find_item([1, 2, 3, 4, 5], 3)
find_item([1, 2, 3, 4, 5], 99)

In [ ]:
# try...else: runs only if NO exception was raised
import json

def safe_parse(text):
    try:
        data = json.loads(text)
    except json.JSONDecodeError as e:
        print(f"Parse error: {e.msg}")
    else:
        # Only runs if parsing succeeded -- keeps success logic
        # separate from the guarded code (EAFP style)
        print(f"Parsed successfully: {data}")
        return data

safe_parse('{"name": "Alice", "age": 30}')
safe_parse("not valid json")

In [ ]:
# Practical: searching with for...else
def first_prime_above(n):
    """Find the first prime number greater than n."""
    candidate = n + 1
    while True:
        for divisor in range(2, int(candidate ** 0.5) + 1):
            if candidate % divisor == 0:
                break  # not prime
        else:
            # No divisor found -> candidate is prime
            return candidate
        candidate += 1

print(f"First prime above 10: {first_prime_above(10)}")
print(f"First prime above 100: {first_prime_above(100)}")

---
## Try It Yourself

### Exercise 1: Indentation Context Manager
Create a context manager `indented(level)` using `@contextmanager` that temporarily modifies a global indent level. Print indented text inside the with block.

In [ ]:
# Exercise 1
import contextlib

_indent_level = 0

@contextlib.contextmanager
def indented(level=1):
    global _indent_level
    _indent_level += level
    try:
        yield _indent_level
    finally:
        _indent_level -= level

def iprint(*args, **kwargs):
    """Print with current indentation."""
    prefix = "  " * _indent_level
    print(prefix, *args, **kwargs)

iprint("Top level")
with indented():
    iprint("Level 1")
    with indented():
        iprint("Level 2")
    iprint("Back to level 1")
iprint("Back to top")

### Exercise 2: Command Dispatcher with match/case
Write a function that takes command tuples like `("greet", "Alice")`, `("add", 3, 4)`, `("quit",)` and dispatches them using match/case.

In [ ]:
# Exercise 2
def dispatch(command):
    match command:
        case ("greet", str(name)):
            return f"Hello, {name}!"
        case ("add", int(a) | float(a), int(b) | float(b)):
            return f"{a} + {b} = {a + b}"
        case ("repeat", str(text), int(n)) if n > 0:
            return text * n
        case ("quit",):
            return "Goodbye!"
        case _:
            return f"Unknown command: {command}"

commands = [
    ("greet", "Alice"),
    ("add", 10, 20),
    ("repeat", "ha", 3),
    ("quit",),
    ("unknown",),
]
for cmd in commands:
    print(f"{str(cmd):30} -> {dispatch(cmd)}")

---
## Key Takeaways

1. **Context managers** (`with` statement) guarantee cleanup via `__enter__`/`__exit__`, even when exceptions occur.

2. The **context manager object** and the **as value** can be different -- `__enter__` returns the `as` value.

3. **`@contextlib.contextmanager`** is the simplest way to create a context manager: code before `yield` = setup, code after = teardown. Always use `try/finally` around the `yield`.

4. **`contextlib`** provides `suppress`, `redirect_stdout`, `closing`, `nullcontext`, and `ExitStack` for common patterns.

5. **match/case** supports class patterns, OR-patterns (`|`), guards (`if`), and nested destructuring -- ideal for interpreters and DSLs.

6. **for/else** and **while/else**: the `else` block runs only if the loop completed without `break`.

7. **try/else**: the `else` block runs only if no exception was raised -- use it to separate guarded code from follow-up logic (EAFP style).

---
## See Also

- [[context-managers-and-with]] -- The with statement and __enter__/__exit__ protocol
- [[contextlib-utilities]] -- closing, suppress, nullcontext, ExitStack
- [[contextmanager-decorator]] -- Building context managers from generators
- [[pattern-matching-beyond-sequences]] -- Class patterns, guards, and the lis.py case study
- [[or-patterns]] -- OR-patterns in match/case
- [[else-blocks-beyond-if]] -- else on for, while, and try

**Next:** Chapter 19 explores concurrency models in Python.